# 02. Veri Ön İşleme (Preprocessing)

Bu notebookta, verideki ölçek uyumsuzluklarını gidermek için `Amount` ve `Time` değişkenleri `RobustScaler` kullanılarak ölçeklendirilmiş ve ardından model eğitiminde kullanılmak üzere veri seti `X_train`, `X_test`, `y_train` ve `y_test` kümelerine, sınıf dağılımı korunarak (`stratify=y`) ayrılmıştır.

In [3]:
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import os

import warnings
warnings.filterwarnings('ignore')

## 1. Veri Okuma

In [4]:
df = pd.read_csv('../creditcard.csv')
print("Veri Boyutu:", df.shape)

Veri Boyutu: (284807, 31)


## 2. Özellik (Feature) Ölçeklendirme
`V1`'den `V28`'e kadar olan özellikler zaten PCA (Temel Bileşen Analizi) ile dönüştürüldükleri için ölçeklendirilmişlerdir. Ancak `Time` ve `Amount` özellikleri orijinal değerlerindedir.
Özellikle `Amount` (İşlem Tutarı) sütununda çok fazla sapan değer (outlier) bulunduğu için, bu tür sapan değerlere karşı dayanıklı olan `RobustScaler` kullanımı tercih edilmiştir.

In [5]:
rob_scaler = RobustScaler()

df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1, 1))

df.drop(['Time', 'Amount'], axis=1, inplace=True)

# Ölçeklendirilmiş yeni sütunları verinin en başına alalım (görsel olarak daha rahat incelemek için)
scaled_amount = df['scaled_amount']
scaled_time = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

display(df.head())

,scaled_amount,scaled_time,V1,V2,V3,V4,V5,V6,V7,V8,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Class
0,1.783274,-0.994983,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0
1,-0.269825,-0.994983,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0
2,4.983721,-0.994972,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0
3,1.418291,-0.994972,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0
4,0.670579,-0.994960,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0


## 3. Eğitim ve Test Verisine Ayırma (Train-Test Split)
Hedef değişkendeki dengesizliği (Sınıf 1 çok az) koruyabilmek amacıyla `stratify=y` özelliği kullanılarak veri %80 Eğitim (Train), %20 Test (Test) olarak ikiye bölünmüştür.

In [6]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train boyutu:", X_train.shape)
print("X_test boyutu:", X_test.shape)
print("y_train boyutu:", y_train.shape)
print("y_test boyutu:", y_test.shape)

# Class dağılımı kontrolü:
print("\nTrain set Class yüzdesi:\n", y_train.value_counts(normalize=True))
print("Test set Class yüzdesi:\n", y_test.value_counts(normalize=True))

X_train boyutu: (227845, 30)
X_test boyutu: (56962, 30)
y_train boyutu: (227845,)
y_test boyutu: (56962,)

Train set Class yüzdesi:
 Class
0    0.998271
1    0.001729
Name: proportion, dtype: float64
Test set Class yüzdesi:
 Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64


## 4. Ön İşlenmiş Verinin Kaydedilmesi
Elde edilen eğitim ve test kümeleri, son aşama olan Modelleme (Modelling) aşamasında kullanılmak üzere dosya olarak çıkartılmıştır.

In [7]:
X_train.to_csv('../outputs/tables/X_train.csv', index=False)
X_test.to_csv('../outputs/tables/X_test.csv', index=False)
y_train.to_csv('../outputs/tables/y_train.csv', index=False)
y_test.to_csv('../outputs/tables/y_test.csv', index=False)
print("Veriler outputs/tables/ klasörüne başarıyla kaydedildi.")

Veriler outputs/tables/ klasörüne başarıyla kaydedildi.
